# Stage 02 — Data Preparation

Load raw data, merge datasets, construct variables, write `data/dataset.parquet`.
Follow `skills/stage_02.md` for detailed instructions.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd

spec = json.loads(PAPER_SPEC.read_text())
print('Identification type:', spec['identification']['type'])
print('Primary file:', spec['identification']['primary_data_file'])
print('Secondary datasets:', spec['identification'].get('secondary_datasets', []))
print('Functional form:', spec['identification']['functional_form'])
print('Fixed effects:', spec['identification']['fixed_effects'])

Identification type: OLS
Primary file: data_taxes.csv
Secondary datasets: []
Functional form: linear
Fixed effects: []


In [2]:
# --- Load primary dataset (CSV) ---
primary_file = RAW_DATA_DIR / spec['identification']['primary_data_file']
df = pd.read_csv(str(primary_file))
print(f'Primary dataset shape: {df.shape}')
print(f'\nColumn names ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col:40s}  dtype={df[col].dtype}')
print()
df.head()

Primary dataset shape: (85, 35)

Column names (35):
  Unnamed: 0                                dtype=int64
  other_taxes                               dtype=float64
  nentryrateav                              dtype=float64
  nbusinessdensitycf                        dtype=float64
  labor                                     dtype=float64
  FDI2005                                   dtype=float64
  Investment2005                            dtype=float64
  dew2004                                   dtype=float64
  dblegor                                   dtype=str
  incomegroup                               dtype=str
  payments2004                              dtype=int64
  time2004                                  dtype=int64
  sb_proc2004                               dtype=float64
  emplo_i2004                               dtype=int64
  code                                      dtype=str
  country                                   dtype=str
  vatsales                                  

,Unnamed: 0,other_taxes,nentryrateav,nbusinessdensitycf,labor,FDI2005,Investment2005,dew2004,dblegor,incomegroup,...,statutory,effective,effective_5yr,EqSec_GDP_2003,MEInv_servi,MEInv_manuf,FDIoecd2004,lngdppc2003,lntime2004,lnpayments2004
0,1,14.514641,7.45706,1.71655,19.506329,2.215858,18.586861,190.250000,French,UpperMiddleIncome,...,35.000000,23.535654,23.797453,34.619999,0.310096,1.265823,NaN,8.843967,6.363028,3.526361
1,2,0.531650,7.54658,5.68313,16.002501,5.230173,25.390350,NaN,French,LowerMiddleIncome,...,20.000000,11.483044,12.455317,NaN,NaN,NaN,NaN,6.790908,7.021084,3.912023
2,3,0.237850,11.26848,13.98344,11.383620,0.788673,25.354759,140.699997,English,HighIncome,...,30.000000,21.964344,23.034048,101.570000,NaN,NaN,3.967628,10.017021,4.672829,2.484907
3,4,0.189130,4.48299,4.90632,25.108360,2.350833,21.050421,248.050003,German,HighIncome,...,34.000000,20.858095,21.041498,20.549999,NaN,NaN,1.520390,10.112533,5.605802,2.995732
4,5,0.538450,7.18533,4.83340,24.267250,10.727272,19.173120,123.050003,French,HighIncome,...,33.990002,16.706244,19.571424,55.110001,NaN,NaN,9.383154,10.049149,5.075174,2.302585


In [3]:
# --- Clean up: drop the R-generated row-index column ---
# The CSV has an 'Unnamed: 0' column that is just a row number from R's write.csv()
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Dropped 'Unnamed: 0' (R row-index column)")

# No secondary datasets to merge for this paper
print(f'No secondary datasets to merge.')
print(f'Shape after cleanup: {df.shape}')

Dropped 'Unnamed: 0' (R row-index column)
No secondary datasets to merge.
Shape after cleanup: (85, 34)


In [4]:
# --- Construct variables ---
# Functional form is linear, no fixed effects, no squared terms needed.
# The paper uses variables as-is from the dataset.
#
# Verify all required variables exist in the dataframe:
# - Primary outcome: Investment2005
# - Additional outcomes: FDI2005, nbusinessdensitycf, nentryrateav
# - Treatment variables: statutory, effective, effective_5yr
# - Controls: 12 variables listed in paper_spec.json

all_outcomes = [spec['identification']['outcome_variable']] + \
               [o['variable'] for o in spec['identification'].get('additional_outcomes', [])]
all_treatments = [spec['identification']['treatment_variable']] + \
                 [t['variable'] for t in spec['identification'].get('additional_treatments', [])]
controls = spec['identification']['controls']

# De-duplicate (primary outcome/treatment are also in the additional lists)
all_outcomes = list(dict.fromkeys(all_outcomes))
all_treatments = list(dict.fromkeys(all_treatments))

print('Outcome variables:', all_outcomes)
print('Treatment variables:', all_treatments)
print('Control variables:', controls)

# Check all exist
missing_cols = [c for c in all_outcomes + all_treatments + controls if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing columns in dataset: {missing_cols}')
else:
    print(f'\nAll {len(all_outcomes + all_treatments + controls)} required variables found in dataset.')

# No variable construction needed — all variables are directly available
# and the functional form is linear with no fixed effects.

Outcome variables: ['Investment2005', 'FDI2005', 'nbusinessdensitycf', 'nentryrateav']
Treatment variables: ['effective_5yr', 'statutory', 'effective']
Control variables: ['other_taxes', 'vatsales', 'pit2004', 'lnpayments2004', 'lngdppc2003', 'propertyrights', 'sb_proc2004', 'emplo_i2004', 'freedomtotradeinternationally', 'seign2004', 'inf10yearavg', 'index_gcr']

All 19 required variables found in dataset.


In [5]:
# --- Validate ---
# Check missingness for all key analysis variables
key_vars = all_outcomes + all_treatments + controls

print('='*60)
print('MISSINGNESS SUMMARY')
print('='*60)
miss = df[key_vars].isnull().sum()
for v in key_vars:
    n_miss = miss[v]
    n_valid = len(df) - n_miss
    tag = ' ***' if n_miss > 0 else ''
    print(f'  {v:40s}  {n_valid:3d}/{len(df)}  ({n_miss} missing){tag}')

print(f'\n--- Complete cases (all key vars non-missing): '
      f'{df[key_vars].dropna().shape[0]} / {len(df)}')

# Per-specification sample sizes after listwise deletion:
# The paper reports different N for different outcomes because missingness varies.
for outcome in all_outcomes:
    vars_for_spec = [outcome] + all_treatments + controls
    n_complete = df[vars_for_spec].dropna().shape[0]
    print(f'  {outcome:30s}: N={n_complete} (with all treatments & controls)')

# Assert minimum sample size for primary specification
primary_vars = [spec['identification']['outcome_variable'],
                spec['identification']['treatment_variable']] + controls
n_primary = df[primary_vars].dropna().shape[0]
assert n_primary >= 30, f'Too few complete obs for primary spec: {n_primary}'
print(f'\nPrimary specification complete cases: {n_primary} (>= 30 required)')

print('\n' + '='*60)
print('DESCRIPTIVE STATISTICS — Key variables')
print('='*60)
df[key_vars].describe().round(3)

MISSINGNESS SUMMARY
  Investment2005                             85/85  (0 missing)
  FDI2005                                    84/85  (1 missing) ***
  nbusinessdensitycf                         80/85  (5 missing) ***
  nentryrateav                               62/85  (23 missing) ***
  effective_5yr                              85/85  (0 missing)
  statutory                                  85/85  (0 missing)
  effective                                  85/85  (0 missing)
  other_taxes                                85/85  (0 missing)
  vatsales                                   85/85  (0 missing)
  pit2004                                    85/85  (0 missing)
  lnpayments2004                             85/85  (0 missing)
  lngdppc2003                                85/85  (0 missing)
  propertyrights                             85/85  (0 missing)
  sb_proc2004                                84/85  (1 missing) ***
  emplo_i2004                                85/85  (0 missing)
  f

,Investment2005,FDI2005,nbusinessdensitycf,nentryrateav,effective_5yr,statutory,effective,other_taxes,vatsales,pit2004,lnpayments2004,lngdppc2003,propertyrights,sb_proc2004,emplo_i2004,freedomtotradeinternationally,seign2004,inf10yearavg,index_gcr
count,85.000,84.000,80.000,62.000,85.000,85.000,85.000,85.000,85.000,85.000,85.000,85.000,85.000,84.000,85.000,81.000,81.000,84.000,64.000
mean,21.457,3.360,5.051,8.113,19.497,29.036,17.437,1.713,17.104,33.498,3.314,8.075,54.000,9.190,37.188,7.149,6.408,13.015,3.430
std,4.764,3.158,4.010,3.325,6.225,6.822,6.711,2.776,8.316,11.286,0.763,1.607,23.258,3.521,17.452,0.939,3.564,20.507,1.141
min,11.204,-2.139,0.004,1.856,6.631,12.500,0.000,0.000,0.000,0.000,1.099,4.987,10.000,2.000,0.000,3.100,0.000,-0.892,1.900
25%,18.183,1.219,1.858,5.789,15.091,25.000,13.736,0.238,14.142,27.500,2.833,6.791,30.000,7.000,26.000,6.500,3.741,2.722,2.600
50%,21.050,2.696,4.602,7.823,20.330,30.000,18.104,0.700,17.500,34.000,3.497,8.226,50.000,9.000,38.000,7.200,5.801,5.661,3.100
75%,24.069,4.168,6.765,10.426,23.450,34.000,21.964,2.159,20.000,40.000,3.871,9.617,70.000,11.000,50.000,7.800,8.184,14.234,4.025
max,40.808,16.440,15.776,16.383,39.873,45.196,39.873,17.576,73.536,60.000,4.585,10.556,90.000,17.000,79.000,9.500,17.482,118.427,6.300


In [6]:
# --- Write output ---
# Save the full dataset (all original columns minus the R row-index) as parquet.
# No rows are dropped — downstream stages handle listwise deletion per specification.
df.to_parquet(str(DATASET_PATH), index=False)
print(f'Written: {DATASET_PATH}')
print(f'  Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'  Columns: {list(df.columns)}')

Written: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\data\dataset.parquet
  Shape: 85 rows x 34 columns
  Columns: ['other_taxes', 'nentryrateav', 'nbusinessdensitycf', 'labor', 'FDI2005', 'Investment2005', 'dew2004', 'dblegor', 'incomegroup', 'payments2004', 'time2004', 'sb_proc2004', 'emplo_i2004', 'code', 'country', 'vatsales', 'freedomtotradeinternationally', 'propertyrights', 'pit2004', 'seign2004', 'legalform', 'index_gcr', 'inf10yearavg', 'avinformal3', 'statutory', 'effective', 'effective_5yr', 'EqSec_GDP_2003', 'MEInv_servi', 'MEInv_manuf', 'FDIoecd2004', 'lngdppc2003', 'lntime2004', 'lnpayments2004']
